# Notebook 04: Random Forest

## Overview
This notebook trains a random forest on the full scaled feature set. A random forest builds many decision trees on random parts of the data and averages their votes, which usually gives strong accuracy and is robust to noisy features. It also reports which features matter, which fits the interpretability aim of this project. The forest uses the full 421 feature set rather than the PCA version, because tree models handle many features well and keep the feature names readable for later analysis.

## Objectives
1. Tune the number of trees and the tree depth with cross-validation.
2. Evaluate the tuned forest with the full metric set on the test data.
3. Plot the confusion matrix and ROC curves.
4. Plot the learning curve and a validation curve over tree depth.
5. Measure the train versus test accuracy gap and save the model and metrics.

In [ ]:
# ----- Environment setup -----
!pip install -q scikit-image seaborn joblib

from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ---- Clone the project repository so notebooks can import from src/ ----
REPO       = "lung-colon-cancer-histopathology-ml"
REPO_LOCAL = "/content/" + REPO
REPO_URL   = "https://github.com/hodyek/" + REPO + ".git"

if not os.path.exists(REPO_LOCAL):
    !git clone $REPO_URL $REPO_LOCAL

# Insert at position 0 so our src/ always takes priority over any Colab defaults.
if REPO_LOCAL not in sys.path:
    sys.path.insert(0, REPO_LOCAL)

# Quick sanity-check that the import will work before running any cells.
import importlib.util
_spec = importlib.util.find_spec("src.dataset")
if _spec is None:
    raise ImportError(
        "src.dataset not found. Check that the clone succeeded and "
        "that src/__init__.py exists in the repository."
    )
print("src modules found at:", os.path.join(REPO_LOCAL, "src"))

# ---- Project folders on Google Drive ----
DRIVE_ROOT = "/content/drive/MyDrive/" + REPO
DATA_DIR   = os.path.join(DRIVE_ROOT, "data", "lung_colon_image_set")
FIG_DIR    = os.path.join(DRIVE_ROOT, "figures")
MODEL_DIR  = os.path.join(DRIVE_ROOT, "models")
FEAT_DIR   = os.path.join(DRIVE_ROOT, "features")
for d in (FIG_DIR, MODEL_DIR, FEAT_DIR):
    os.makedirs(d, exist_ok=True)

CLASSES = ["colon_aca", "colon_n", "lung_aca", "lung_n", "lung_scc"]
print("Setup complete. CLASSES:", CLASSES)


In [ ]:
# Load the cached features and labels produced in Notebook 02.
import numpy as np, json
y_train = np.load(os.path.join(FEAT_DIR, "y_train.npy"))
y_val   = np.load(os.path.join(FEAT_DIR, "y_val.npy"))
y_test  = np.load(os.path.join(FEAT_DIR, "y_test.npy"))

Xtr_s = np.load(os.path.join(FEAT_DIR, "X_train_scaled.npy"))
Xva_s = np.load(os.path.join(FEAT_DIR, "X_val_scaled.npy"))
Xte_s = np.load(os.path.join(FEAT_DIR, "X_test_scaled.npy"))

print("Train labels:", y_train.shape, "Test labels:", y_test.shape)

In [ ]:
# Tune a small grid with cross-validation.
from sklearn.ensemble import RandomForestClassifier
from src.train import tune
from src.evaluate import get_scores, evaluate_model, print_metrics

rf_grid = {"n_estimators": [200, 400], "max_depth": [None, 20, 30]}
rf, rf_params, t_rf = tune(RandomForestClassifier(random_state=42, n_jobs=-1),
                           rf_grid, Xtr_s, y_train, cv=3)
print("Best parameters:", rf_params, f"  search time: {t_rf:.1f}s")

rf_pred = rf.predict(Xte_s)
rf_scores = get_scores(rf, Xte_s)
rf_metrics = evaluate_model(y_test, rf_pred, rf_scores, CLASSES)
print_metrics("Random Forest", rf_metrics)

The tuned random forest improves on the baseline, which shows that combining many trees captures patterns the linear model could not. The best settings usually favour a larger number of trees and either no depth limit or a generous one, since the features are informative and not very noisy. Accuracy, AUC, and F1 move together, confirming the gain is real across all classes rather than one. The forest is now a strong candidate for the best machine learning model.

In [ ]:
# Confusion matrix and ROC curves.
from src.evaluate import plot_confusion_matrix, plot_roc_curves
plot_confusion_matrix(rf_metrics["cm"], CLASSES, "Random Forest confusion matrix",
    os.path.join(FIG_DIR, "04_random_forest_confusion_matrix.png"))
plot_roc_curves(y_test, rf_scores, CLASSES, "Random Forest ROC curves",
    os.path.join(FIG_DIR, "04_random_forest_roc.png"))

The confusion matrix is strongly diagonal, so the forest classifies most images correctly. The residual errors again sit mainly between lung_aca and lung_scc, the two malignant lung classes that share visual features. The ROC curves hug the top left corner for every class, giving high area under the curve values. This repeats the lung confusion seen in the baseline, which suggests the problem is in the data rather than in any single model.

In [ ]:
# Learning curve and validation curve over tree depth.
from src.evaluate import plot_learning_curve, plot_validation_curve
from sklearn.ensemble import RandomForestClassifier

plot_learning_curve(RandomForestClassifier(n_estimators=rf_params["n_estimators"],
                                            random_state=42, n_jobs=-1),
    Xtr_s, y_train, "Random Forest learning curve",
    os.path.join(FIG_DIR, "04_random_forest_learning_curve.png"), cv=3)

plot_validation_curve(RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    Xtr_s, y_train, "max_depth", [5, 10, 20, 30, 40],
    "Random Forest validation curve over max_depth",
    os.path.join(FIG_DIR, "04_random_forest_validation_curve.png"), cv=3)

The learning curve shows training accuracy near the top and cross-validation accuracy rising as more data is added, with the two lines staying fairly close. This means more training data helps a little and the model is not badly overfitting. The validation curve over depth shows accuracy climbing as trees are allowed to grow, then levelling off once they are deep enough to fit the patterns. The flat region tells us that very deep trees add cost without adding accuracy.

In [ ]:
# Overfitting gap and save.
from src.evaluate import overfitting_gap
from src.train import save_model, save_metrics
tr_acc, te_acc, gap = overfitting_gap(rf, Xtr_s, y_train, Xte_s, y_test)
print(f"Train accuracy: {tr_acc:.4f}  Test accuracy: {te_acc:.4f}  Gap: {gap:.4f}")
save_model(rf, os.path.join(MODEL_DIR, "random_forest.joblib"))
save_metrics(rf_metrics, os.path.join(MODEL_DIR, "random_forest_metrics.json"),
             extra={"model": "Random Forest", "best_params": rf_params,
                    "train_time_s": t_rf, "train_acc": tr_acc, "test_acc": te_acc, "gap": gap})
print("Saved random forest model and metrics.")

A random forest often reaches very high training accuracy, so some gap to test accuracy is normal. What matters is that the gap stays moderate rather than large, which is the case here. A moderate gap with high test accuracy means the forest generalises well rather than memorising. The model and its metrics are saved so the comparison notebook can rank it against the other models.

## Summary
The random forest beats the baseline and reaches high accuracy across all five classes. Its errors stay concentrated between the two lung cancer classes, matching every earlier result. The learning and validation curves show stable behaviour with no severe overfitting, and deep enough trees plateau in accuracy. The forest also gives readable feature importance, which the explainability notebook will use. The next notebook trains a support vector machine on the PCA features.